In [ ]:
import pandas as pd
from collections import Counter

print("Imports successful")

In [63]:
PIECES = 'PNBRQpnbrq'  # uppercase = white, lowercase = black, king omitted
VALUES = {'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9} # piece values

def process_csv(file: str):
    print("Importing: " + file)
    csv = pd.read_csv(file) # read CSV file

    rows = [] # create list
    for fen in csv['FEN']: # for each row fen in CSV file
        placement, player, castling, _ep, half, full = fen.split(' ') # split fen into substrings separated by a ' ' and give them a name
        counts = Counter(placement) # counts occurrences of every character inside the string
        rows.append( # add parsed row
            [counts.get(p, 0) for p in PIECES] # add counts of pieces
            + [int(player == 'w')] # add player as int (White = 1, Black = 0)
            + [int(c in castling) for c in 'KQkq'] # add all 4 castling rules as ints (1 = true, 0 = false)
            + [int(half), int(full)] # add number of half moves for stalemate and full moves played
        )

    # convert standard list to dataframe
    new_data = pd.DataFrame(
        rows, # the data from above
        columns=list(PIECES) + [ # piece column labels by piece, then other labels corresponding to the data above
            'white_to_move',
            'white_kingside', 'white_queenside', 'black_kingside', 'black_queenside',
            'halfMoves', 'fullMoves',
        ],
    )
    new_data['Evaluation'] = csv['Evaluation'].values # add evaluation to csv, could process further to convert mate values into int

    # add up material value and add to dataframe
    new_data['white_material'] = sum(new_data[p] * v for p, v in VALUES.items())
    new_data['black_material'] = sum(new_data[p.lower()] * v for p, v in VALUES.items())
    new_data['material_diff'] = new_data['white_material'] - new_data['black_material'] # add material difference to data frame

    new_data.to_csv(file[:-4] + 'Formatted.csv', index=False) # export dataframe to csv
    print(file + " imported to " + file[:-4] + 'Formatted.csv')

In [64]:
process_csv('chessData.csv')
process_csv('random_evals.csv')
process_csv('tactic_evals.csv')

Importing: chessData.csv
chessData.csvimported to chessDataFormatted.csv
Importing: random_evals.csv
random_evals.csvimported to random_evalsFormatted.csv
Importing: tactic_evals.csv
tactic_evals.csvimported to tactic_evalsFormatted.csv
